# Movie Recommender: Content-Based vs Collaborative Filtering

Goal: build both approaches, evaluate, and decide which to serve from the API (or combine into a hybrid).

**Dataset**: [The Movies Dataset](https://www.kaggle.com/rounakbanik/the-movies-dataset) (Kaggle, CC0) — richer than plain MovieLens: 45k movies with overview text, genres, keywords, cast/crew, and TMDb poster paths already included, plus a ratings file for collaborative filtering.

**Setup**
1. `kaggle datasets download -d rounakbanik/the-movies-dataset -p ../data/the-movies-dataset --unzip` (requires a Kaggle API token at `~/.kaggle/`)
2. You should now have `movies_metadata.csv`, `keywords.csv`, `credits.csv`, `links_small.csv`, `ratings_small.csv` in `backend/data/the-movies-dataset/`
3. We use `ratings_small.csv` (100k ratings, MovieLens-100k under the hood) instead of the full `ratings.csv` (26M rows, 700MB) for fast local iteration

## 1. Load data

In [ ]:
import pandas as pd

DATA_DIR = "../data/the-movies-dataset"

movies = pd.read_csv(f"{DATA_DIR}/movies_metadata.csv", low_memory=False)

# 3 rows have shifted/corrupted columns in the source CSV (id ends up as a date) -- drop them
movies = movies[pd.to_numeric(movies["id"], errors="coerce").notna()].copy()
movies["id"] = movies["id"].astype(int)
movies = movies.drop_duplicates(subset="id")

# keywords.csv and credits.csv both have duplicate `id` rows in the raw data
# (987 and 44 respectively) -- dedupe before merging or the inner join
# silently multiplies those movies' rows (cartesian blowup)
keywords = pd.read_csv(f"{DATA_DIR}/keywords.csv").drop_duplicates(subset="id")
credits = pd.read_csv(f"{DATA_DIR}/credits.csv").drop_duplicates(subset="id")
links_small = pd.read_csv(f"{DATA_DIR}/links_small.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings_small.csv")

movies = movies.merge(keywords, on="id").merge(credits, on="id")
movies = movies.reset_index(drop=True)
movies[["id", "title", "genres", "overview"]].head()

## 2. Content-based filtering

Build a "soup" of each movie's genres, keywords, top-3 cast, and director, then rank by cosine similarity to the query title.

Note: we deliberately do *not* materialize a full N×N similarity matrix — with 45k movies that's a ~16GB dense array and reliably crashes on a laptop. Instead we compare the query row against all rows on demand, which only needs a single 1×N slice.

In [2]:
import ast


def parse_names(cell: str, key: str = "name", top_n: int | None = None) -> list[str]:
    try:
        items = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return []
    names = [item[key] for item in items]
    return names[:top_n] if top_n else names


def get_director(crew_cell: str) -> str:
    for member in ast.literal_eval(crew_cell):
        if member["job"] == "Director":
            return member["name"]
    return ""


def clean_token(s: str) -> str:
    return str(s).lower().replace(" ", "")


movies["genre_names"] = movies["genres"].apply(parse_names)
movies["keyword_names"] = movies["keywords"].apply(parse_names)
movies["cast_names"] = movies["cast"].apply(lambda x: parse_names(x, top_n=3))
movies["director"] = movies["crew"].apply(get_director)

movies["soup"] = movies.apply(
    lambda row: " ".join(
        [clean_token(g) for g in row["genre_names"]]
        + [clean_token(k) for k in row["keyword_names"]]
        + [clean_token(c) for c in row["cast_names"]]
        + [clean_token(row["director"])] * 3  # weight director higher
    ),
    axis=1,
)

movies["soup"].iloc[0]

'animation comedy family jealousy toy boy friendship friends rivalry boynextdoor newtoy toycomestolife tomhanks timallen donrickles johnlasseter johnlasseter johnlasseter'

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

count = CountVectorizer(stop_words="english")
count_matrix = count.fit_transform(movies["soup"])  # sparse, (n_movies, vocab_size)

title_to_idx = pd.Series(movies.index, index=movies["title"]).drop_duplicates()


def content_based_recommend(title: str, n: int = 10):
    idx = title_to_idx.get(title)
    if idx is None:
        raise ValueError(f"'{title}' not found")
    sims = cosine_similarity(count_matrix[idx], count_matrix).flatten()
    ranked = sims.argsort()[::-1]
    top = [i for i in ranked if i != idx][:n]
    return movies.iloc[top][["id", "title", "genre_names"]]


content_based_recommend("Toy Story")

,id,title,genre_names
19185,13927,Tin Toy,[Animation]
19239,13926,Red's Dream,[Animation]
3008,863,Toy Story 2,"[Animation, Comedy, Family]"
19289,13928,Knick Knack,[Animation]
10698,13925,Luxo Jr.,[Animation]
17435,49013,Cars 2,"[Animation, Family, Adventure, Comedy]"
22926,13934,Mater and the Ghostlight,"[Animation, Family]"
11018,920,Cars,"[Animation, Adventure, Comedy, Family]"
2250,9487,A Bug's Life,"[Adventure, Animation, Comedy, Family]"
29070,96872,Superstar Goofy,"[Animation, Comedy, Family]"


## 3. Collaborative filtering (matrix factorization)

Use `implicit`'s ALS on the user-item ratings matrix. `implicit` is used here instead of `scikit-surprise` because `scikit-surprise` doesn't build on Python 3.13 (its Cython extension is unmaintained for newer Python).

`ratings_small.csv` uses MovieLens `movieId`, but `movies_metadata.csv` is keyed on TMDb `id` — `links_small.csv` maps between them, so we join through it first.

In [4]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

links_map = links_small.dropna(subset=["tmdbId"]).set_index("movieId")["tmdbId"].astype(int)

ratings_mapped = ratings.copy()
ratings_mapped["tmdbId"] = ratings_mapped["movieId"].map(links_map)
ratings_mapped = ratings_mapped.dropna(subset=["tmdbId"])
ratings_mapped["tmdbId"] = ratings_mapped["tmdbId"].astype(int)

user_cat = ratings_mapped["userId"].astype("category")
movie_cat = ratings_mapped["tmdbId"].astype("category")

user_item = sp.csr_matrix(
    (ratings_mapped["rating"], (user_cat.cat.codes, movie_cat.cat.codes))
)

tmdb_id_categories = movie_cat.cat.categories  # inner index -> tmdbId

als = AlternatingLeastSquares(factors=50, regularization=0.1, iterations=20, random_state=42)
als.fit(user_item)

  0%|          | 0/20 [00:00<?, ?it/s]

In [5]:
def collaborative_similar_items(tmdb_id: int, n: int = 10):
    """Item-item similarity from the learned ALS latent factors."""
    inner_id = tmdb_id_categories.get_loc(tmdb_id)
    similar_ids, scores = als.similar_items(inner_id, N=n + 1)
    similar_ids = similar_ids[similar_ids != inner_id][:n]
    top_tmdb_ids = [tmdb_id_categories[i] for i in similar_ids]
    return movies[movies["id"].isin(top_tmdb_ids)][["id", "title", "genre_names"]]


sample_tmdb_id = int(movies.loc[title_to_idx["Toy Story"], "id"])
collaborative_similar_items(sample_tmdb_id)

,id,title,genre_names
31,63,Twelve Monkeys,"[Science Fiction, Thriller, Mystery]"
256,11,Star Wars,"[Adventure, Action, Science Fiction]"
351,13,Forrest Gump,"[Comedy, Drama, Romance]"
475,329,Jurassic Park,"[Adventure, Science Fiction]"
621,36447,Carried Away,"[Drama, Romance]"
638,954,Mission: Impossible,"[Adventure, Action, Thriller]"
722,664,Twister,"[Action, Adventure, Drama]"
763,602,Independence Day,"[Action, Adventure, Science Fiction]"
1176,1892,Return of the Jedi,"[Adventure, Action, Science Fiction]"
1234,105,Back to the Future,"[Adventure, Comedy, Science Fiction, Family]"


## 4. Evaluate and decide

- Compare qualitative relevance of content-based neighbors (should share genre/theme/cast) vs. collaborative "people who rated this also rated" neighbors
- Note: `ratings_small` only covers ~9k of the 45k movies (671 users), so collaborative filtering has a real cold-start gap here — content-based can recommend across the full catalog, CF only across movies with enough ratings
- A natural hybrid: use CF similarity when the queried movie has enough ratings, fall back to content-based otherwise
- Write the decision + reasoning into the top-level README
- `movies["poster_path"]` already gives TMDb poster paths directly (`https://image.tmdb.org/t/p/w342{poster_path}`) — no live TMDb API call needed for posters, though the `tmdb.py` service is still useful as a fallback for movies without one
- Once decided, port the winning `recommend()`/`search()` functions into `backend/app/services/recommender.py`